In [1]:
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, countDistinct

In [2]:
MINIO_ENDPOINT = "http://minio.mon-namespace.svc.cluster.local:80"
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "DataL@b_Cae123"
NESSIE_URI = "http://nessie.trino.svc.cluster.local:19120/api/v1" 

conf = (
    SparkConf()
    .setAppName("Job_Transformation_silver")
    .set("spark.driver.host", "127.0.0.1")
    .set("spark.driver.bindAddress", "127.0.0.1")
    .set("spark.driver.extraJavaOptions", "-Dorg.apache.poi.util.IOUtils.setByteArrayMaxOverride=1000000000")
    .set("spark.driver.memory", "16g")
    # AJOUT DES PACKAGES : On ajoute le jar nessie-spark-extensions
    .set("spark.jars.packages", 
         "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1,"
         "org.apache.hadoop:hadoop-aws:3.3.4,"
         "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.5_2.12:0.77.1,""com.crealytics:spark-excel_2.12:3.5.0_0.20.3")
     
     .set("spark.sql.extensions", 
         "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
         "org.projectnessie.spark.extensions.NessieSparkSessionExtensions")

    # CONFIGURATION DU CATALOGUE NESSIE
    .set("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .set("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .set("spark.sql.catalog.nessie.uri", NESSIE_URI)
    .set("spark.sql.catalog.nessie.ref", "main")
    
    # On définit le bucket bronze comme entrepôt par défaut du catalogue
    .set("spark.sql.catalog.nessie.warehouse", "s3a://warehouse/")
    
    # CONFIGURATION MINIO (S3A)
    .set("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .set("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .set("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
    #.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
)

In [3]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()

/usr/local/lib/python3.12/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found


:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12 added as a dependency
com.crealytics#spark-excel_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-aef7fc4a-7edc-4d35-9c52-ceb1e307d5b4;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12;0.77.1 in central
	found com.crealytics#spark-excel_2.12;3.5.0_0.20.3 in central
	found org.apache.poi#poi;5.2.5 in central
	found commons-codec#commons-codec;1.

In [4]:
# Créer la branche gold depuis main
spark.sql("CREATE BRANCH IF NOT EXISTS gold_agregation IN nessie FROM main")
spark.sql("USE REFERENCE gold_agregation IN nessie")

DataFrame[refType: string, name: string, hash: string]

In [5]:
df_silver = spark.sql("SELECT * FROM nessie.silver.extrait_solde")

26/04/13 17:12:12 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [6]:
# Vérification
print(f"Nombre de lignes silver : {df_silver.count()}")
df_silver.printSchema()
df_silver.show(5, truncate=False)

Nombre de lignes silver : 34715
root
 |-- SERVICE: string (nullable = true)
 |-- ORGANISME: string (nullable = true)
 |-- EMPLOI: string (nullable = true)
 |-- GRADE: string (nullable = true)
 |-- CLASSE/ECHELON: string (nullable = true)



+---------------------------+------------------------------------------------------+----------------------------------------+----------------------+----------------------------------+
|SERVICE                    |ORGANISME                                             |EMPLOI                                  |GRADE                 |CLASSE/ECHELON                    |
+---------------------------+------------------------------------------------------+----------------------------------------+----------------------+----------------------------------+
|D P I F R                  |Min. des Eaux et Forêts                               |Ingénieur des Techniques: Eaux et Forêts|Fonctionnaire Grade A3|Classe Exceptionnelle 3ème Echelon|
|E P H D                    |Min. de la Santé, de l'Hygiène Publique et de la C M U|Infirmier Diplomé d'Etat                |Fonctionnaire Grade B3|Classe Exceptionnelle 3ème Echelon|
|Lycée Moderne Harris Adjamé|Min. de l'Education Nationale et de l'Alphabétisati

In [8]:
# Gold 1 : Nombre d'employés par organisme
df_gold_organisme = spark.sql("""
    SELECT 
        ORGANISME,
        COUNT(*) as nb_employes,
        COUNT(DISTINCT EMPLOI) as nb_emplois_distincts,
        COUNT(DISTINCT GRADE) as nb_grades_distincts
    FROM nessie.silver.extrait_solde
    GROUP BY ORGANISME
    ORDER BY nb_employes DESC
""")


In [9]:
# Gold 2 : Nombre d'employés par grade
df_gold_grade = spark.sql("""
    SELECT 
        GRADE,
        COUNT(*) as nb_employes,
        COUNT(DISTINCT ORGANISME) as nb_organismes
    FROM nessie.silver.extrait_solde
    GROUP BY GRADE
    ORDER BY nb_employes DESC
""")

In [10]:
# Gold 3 : Nombre d'employés par service
df_gold_service = spark.sql("""
    SELECT 
        SERVICE,
        ORGANISME,
        COUNT(*) as nb_employes,
        COUNT(DISTINCT EMPLOI) as nb_emplois
    FROM nessie.silver.extrait_solde
    GROUP BY SERVICE, ORGANISME
    ORDER BY nb_employes DESC
""")


In [11]:
# Gold 4 : Répartition par classe/echelon
df_gold_echelon = spark.sql("""
    SELECT 
        `CLASSE/ECHELON`,
        GRADE,
        COUNT(*) as nb_employes
    FROM nessie.silver.extrait_solde
    GROUP BY `CLASSE/ECHELON`, GRADE
    ORDER BY nb_employes DESC
""")

In [12]:
# Créer namespace gold sur la branche courante
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.gold")

DataFrame[]

In [13]:
# Écrire dans gold
df_gold_organisme.write \
    .format("iceberg") \
    .mode("overwrite") \
    .saveAsTable("nessie.gold.employes_par_organisme")

df_gold_grade.write \
    .format("iceberg") \
    .mode("overwrite") \
    .saveAsTable("nessie.gold.employes_par_grade")

df_gold_service.write \
    .format("iceberg") \
    .mode("overwrite") \
    .saveAsTable("nessie.gold.employes_par_service")

df_gold_echelon.write \
    .format("iceberg") \
    .mode("overwrite") \
    .saveAsTable("nessie.gold.employes_par_echelon")



In [14]:
# Merger dans main
spark.sql("MERGE BRANCH gold_agregation INTO main IN nessie")

DataFrame[name: string, hash: string]

In [15]:
# Supprimer la branche
spark.sql("DROP BRANCH gold_agregation IN nessie")


DataFrame[status: string]